# hydra_core smoke test

Tests for `hydra-core` — Facebook's configuration framework built on OmegaConf.

In [ ]:
# Example 1: OmegaConf DictConfig basics (Hydra configs are OmegaConf DictConfig objects)
from omegaconf import OmegaConf, DictConfig

cfg = OmegaConf.create({
    "db": {"driver": "postgres", "host": "localhost", "port": 5432},
    "app": {"name": "demo", "debug": True},
})

assert isinstance(cfg, DictConfig)
assert cfg.db.driver == "postgres"
assert cfg.db.port == 5432
assert cfg.app.debug is True

yaml_text = OmegaConf.to_yaml(cfg)
print(yaml_text)

In [ ]:
# Example 2: hydra.utils.instantiate — build an object from a config _target_ spec.
from hydra.utils import instantiate
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    "_target_": "collections.OrderedDict",
    "_args_": [[("a", 1), ("b", 2), ("c", 3)]],
})

obj = instantiate(cfg)
import collections
assert isinstance(obj, collections.OrderedDict)
assert list(obj.items()) == [("a", 1), ("b", 2), ("c", 3)]
print(f"instantiated: {obj}")

In [ ]:
# Example 3: hydra.utils.get_class / get_method — string-based resolution helpers.
from hydra.utils import get_class, get_method

cls = get_class("collections.OrderedDict")
import collections
assert cls is collections.OrderedDict

fn = get_method("os.path.join")
import os.path
assert fn is os.path.join
assert fn("a", "b", "c") == os.path.join("a", "b", "c")
print(f"get_class: {cls}")
print(f"get_method: {fn}")

In [ ]:
# Example 4: Compose a config programmatically using hydra.initialize_config_dir + compose.
# Use a temp dir with a tiny YAML config to exercise the full Hydra compose API.
import os
import tempfile
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

with tempfile.TemporaryDirectory() as tmpdir:
    cfg_path = os.path.join(tmpdir, "config.yaml")
    with open(cfg_path, "w") as f:
        f.write("foo: 1\nbar: hello\nnested:\n  x: 42\n")

    GlobalHydra.instance().clear()
    with initialize_config_dir(version_base=None, config_dir=tmpdir):
        cfg = compose(config_name="config")

    assert cfg.foo == 1
    assert cfg.bar == "hello"
    assert cfg.nested.x == 42
    print(f"composed cfg: {dict(cfg)}")

In [ ]:
# Example 5: ConfigStore — Hydra's structured config registry. Register, retrieve, assert.
from dataclasses import dataclass
from hydra.core.config_store import ConfigStore

@dataclass
class DBConfig:
    driver: str = "sqlite"
    host: str = "localhost"
    port: int = 5432

cs = ConfigStore.instance()
cs.store(name="db_schema", node=DBConfig)

repo = cs.repo
assert "db_schema.yaml" in repo
entry = repo["db_schema.yaml"]
assert entry.node.driver == "sqlite"
assert entry.node.port == 5432
print(f"ConfigStore entry: name={entry.name}, node={entry.node}")
assert True